# 01 · Parte 1 — Datos, limpieza y validación

**Responsable:** _Jose Angel Morales Nuñez_

**Objetivo:** dejar un dataset limpio y un esquema de validación que las otras dos partes consuman tal cual. Esta parte es la base: sin ella, las partes 2 y 3 no pueden medirse.

Referencia: hallazgos del notebook `00_exploracion.ipynb` (secciones 1, 3, 4, 7 y 8).

Convenciones comunes a las tres partes:

- Rutas relativas a la raíz del repo: `../data/raw/`, `../data/processed/`, `../models/`, `../submissions/`.
- Todo lo que una parte produce para otra se guarda en `../data/processed/<parte>/` en formato parquet, con las columnas documentadas en la sección *Salidas* de cada notebook.
- Identificador de pozo: el prefijo de 8 caracteres del archivo (`000d7d20`). Índice de punto: la fila del CSV horizontal (el mismo que usa `sample_submission`).
- Métrica oficial: RMSE sobre los puntos con índice ≥ PS (Prediction Start), en pies.

## Alcance

1. **Carga unificada**: función que, dado un id de pozo, devuelve el CSV horizontal y el typewell con tipos correctos y el índice del PS.
2. **Limpieza de GR**: tratamiento de NaN del pozo horizontal (huecos de 1-9 ft interpolables; huecos largos marcados con una máscara, no inventados). Decidir y documentar si se suaviza o normaliza el GR y cómo.
3. **Remuestreo del typewell** a una grilla común de TVT (los typewells vienen a 0.25 / 0.5 / 1 ft) y tratamiento de los 13 pozos cuyo typewell no cubre su rango de TVT.
4. **Tabla maestra de pozos**: una fila por pozo con metadatos (n, PS, azimut, coordenadas de cabeza y fin, typewell asignado, formación dominante…).
5. **Esquema de validación**: partición por pozos con criterio espacial (hay vecinos a < 50 ft) en K folds reproducibles. Guardar la asignación pozo → fold.
6. **Función de evaluación**: RMSE por pozo y global sobre el tramo ≥ PS, a partir de un DataFrame de predicciones con el formato acordado.
7. **Formato de predicción**: fijar el esquema que las partes 2 y 3 deben producir y escribir un validador.

## Salidas (contrato con las partes 2 y 3)

| Archivo | Contenido |
|---|---|
| `../data/processed/p1/wells.parquet` | tabla maestra: una fila por pozo (`well`, `n`, `ps`, `azimuth`, `x0`, `y0`, `x_end`, `y_end`, `fold`, …) |
| `../data/processed/p1/horizontal/<well>.parquet` | CSV horizontal limpio: columnas originales + `GR_clean`, `GR_mask` (True = valor interpolado o ausente) |
| `../data/processed/p1/typewell/<well>.parquet` | typewell remuestreado a grilla regular de TVT: `TVT`, `GR`, `Geology` |
| `../data/processed/p1/folds.parquet` | `well`, `fold` |

Formato de predicción que deben respetar las partes 2 y 3:

| Columna | Tipo | Descripción |
|---|---|---|
| `well` | str | id del pozo |
| `idx` | int | fila del CSV horizontal (≥ PS) |
| `tvt_pred` | float | TVT predicho en pies |

Funciones a exponer (copiarlas a un módulo `src/` cuando se estabilicen): `load_well`, `clean_gr`, `resample_typewell`, `make_folds`, `rmse_by_well`, `validate_prediction`.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed/p1")
PROCESSED.mkdir(parents=True, exist_ok=True)

wells = sorted(p.name[:8] for p in (RAW / "train").glob("*__horizontal_well.csv"))
sub = pd.read_csv(RAW / "sample_submission.csv")

### "Verificaciones (se pueden omitir)"

In [ ]:
train = sorted((RAW / "train").rglob("*"))
test = sorted((RAW / "test").rglob("*"))
print("train:", len(train), "elementos")
for p in train[:8]:
    print("  ", p.relative_to(RAW))
print("test:", len(test), "elementos")
for p in test[:8]:
    print("  ", p.relative_to(RAW))

sub = pd.read_csv(RAW / "sample_submission.csv")
print(sub.shape)
print(sub.head())

In [ ]:
well = "000d7d20"
hz = pd.read_csv(RAW / "train" / f"{well}__horizontal_well.csv")
tw = pd.read_csv(RAW / "train" / f"{well}__typewell.csv")

print("horizontal:", hz.shape)
print(hz.dtypes)
print(hz.head())
print("NaN por columna:")
print(hz.isna().sum())

print("typewell:", tw.shape)
print(tw.dtypes)
print(tw.head())

## 2. Carga unificada

In [ ]:
def load_well(well, split="train"):
    """Devuelve (horizontal, typewell, ps) de un pozo."""
    base = RAW / split
    hz = pd.read_csv(base / f"{well}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{well}__typewell.csv")
    falta = hz["TVT_input"].isna()
    ps = int(falta.idxmax()) if falta.any() else len(hz)
    return hz, tw, ps

### "Verificaciones (se pueden omitir)"

In [ ]:
hz, tw, ps = load_well("000d7d20")
print("columnas typewell:", tw.columns.tolist())
print("PS calculado:", ps, "| filas del CSV:", len(hz))
print(hz[["TVT", "TVT_input", "GR"]].isna().sum())

ids = (sub[sub["id"].str.startswith("000d7d20_")]["id"]
       .str.split("_").str[1].astype(int))
print("sample_submission:", ids.min(), "a", ids.max(), "|", len(ids), "filas")

In [ ]:
print(len(wells), "pozos en train")

raros = []
for w in wells:
    hz, tw, ps = load_well(w)
    cola_ok = (hz["TVT_input"].iloc[:ps].notna().all()
               and hz["TVT_input"].iloc[ps:].isna().all())
    if not cola_ok:
        raros.append(w)
print("pozos con TVT_input no contiguo:", len(raros), raros[:5])

hz_t, tw_t, ps_t = load_well("000d7d20", split="test")
print("columnas en test:", hz_t.columns.tolist(), "| PS:", ps_t)

## 3. Limpieza de GR

**Decisiones de limpieza de GR**

- Cada fila equivale a 1 ft (`MD` avanza de 1 en 1 en los 773 pozos), así que huecos en filas = huecos en pies.
- Huecos de 1 a 9 ft: se interpolan linealmente (`GR_clean`). Son el 99.2 % de los huecos y el 93.6 % de las filas sin GR.
- Huecos de 10 ft o más y huecos en los bordes del pozo: no se inventan datos; quedan en NaN en `GR_clean` (96 310 de 1 507 972 filas faltantes). De los 70 huecos de 50 ft o más, 52 están en un borde del pozo.
- `GR_mask = True` donde el GR original faltaba, se haya interpolado o no. Antes del PS falta en promedio el 24 % del GR original (mediana 20 %) y desde el PS el 31 % (mediana 30 %): casi un tercio del tramo evaluado es interpolado, no medido.
- `clean_gr` nunca modifica un valor medido (verificado en los 773 pozos). El límite de 9 ft es el parámetro `max_gap`.
- Escala: el GR del horizontal y el del typewell están en la misma escala (media de las medias 88.1 y 88.7; ningún valor negativo). No se normaliza: `GR_clean` conserva las unidades originales y cada parte puede normalizar como necesite.
- Ruido: la razón std(saltos) / std total va de 0.05 a 0.60 (mediana 0.29) en los 773 pozos, o sea una señal continua. No se suaviza `GR_clean`; `smooth_gr` (mediana móvil centrada, ventana 9) queda como función opcional.
- Valores altos: no se recortan. De 3 pozos revisados con GR > 300, uno es un pico aislado (`a4f989c2`: 487 con vecinos en 118) y los otros dos son zonas de GR alto (vecinos en 315 y 200). `smooth_gr` atenúa los picos aislados.

In [ ]:
def runs_nan(s):
    """Inicio y largo de cada racha de NaN consecutivos."""
    m = s.isna().to_numpy().astype(int)
    d = np.diff(np.concatenate(([0], m, [0])))
    ini = np.where(d == 1)[0]
    fin = np.where(d == -1)[0]
    return ini, fin - ini

# --- un pozo ---
hz, tw, ps = load_well("000d7d20")
print("paso de MD:", hz["MD"].diff().value_counts().head(3).to_dict())
ini, largo = runs_nan(hz["GR"])
print("huecos:", len(ini), "| NaN totales:", largo.sum(), "| PS:", ps)
print(pd.DataFrame({"inicio": ini, "largo": largo}).head(10))

# --- los 773 pozos ---
filas = []
for w in wells:
    hz, tw, ps = load_well(w)
    ini, largo = runs_nan(hz["GR"])
    for i, l in zip(ini, largo):
        filas.append((w, i, l, i >= ps))
huecos = pd.DataFrame(filas, columns=["well", "inicio", "largo", "empieza_tras_ps"])
huecos["tramo"] = pd.cut(huecos["largo"], [0, 9, 49, 199, np.inf],
                         labels=["1-9", "10-49", "50-199", "200+"])
print(len(huecos), "huecos en", huecos["well"].nunique(), "pozos")
print(huecos["tramo"].value_counts().sort_index())
print(huecos.groupby("tramo", observed=True)["largo"].sum())

In [ ]:
def clean_gr(hz, max_gap=9):
    """Interpola huecos cortos de GR y marca todo lo que faltaba.

    GR_clean: GR con los huecos de hasta max_gap filas interpolados (lineal).
              Los huecos más largos, y los que están en los bordes del pozo, siguen en NaN.
    GR_mask : True donde el GR original faltaba (se haya interpolado o no).
    """
    gr = hz["GR"]
    mask = gr.isna()

    # longitud del hueco al que pertenece cada fila vacía
    grupo = (~mask).cumsum()
    largo_hueco = mask.groupby(grupo).transform("sum")

    # interpola solo entre valores válidos (no extrapola en los bordes)
    interp = gr.interpolate(method="linear", limit_area="inside")

    # deja en NaN lo que cae en huecos largos
    gr_clean = interp.where(~(mask & (largo_hueco > max_gap)))

    out = hz.copy()
    out["GR_clean"] = gr_clean
    out["GR_mask"] = mask
    return out

In [ ]:
def smooth_gr(gr_clean, window=9):
    """Mediana móvil centrada. No rellena los huecos que quedaron en NaN."""
    suave = gr_clean.rolling(window, center=True, min_periods=1).median()
    return suave.where(gr_clean.notna())

### "Verificaciones y análisis (se pueden omitir)"

In [ ]:
mal_md, nan_orig, nan_clean, cambiados = [], 0, 0, 0
for w in wells:
    hz, tw, ps = load_well(w)
    if not hz["MD"].diff().dropna().eq(1.0).all():
        mal_md.append(w)
    out = clean_gr(hz)
    m = out["GR_mask"]
    nan_orig += m.sum()
    nan_clean += out["GR_clean"].isna().sum()
    if not (out.loc[~m, "GR"] == out.loc[~m, "GR_clean"]).all():
        cambiados += 1

print("pozos con MD no uniforme:", len(mal_md))
print("NaN originales:", nan_orig, "| NaN tras limpiar:", nan_clean)
print("rellenados:", nan_orig - nan_clean)
print("pozos donde cambió un valor original:", cambiados)

In [ ]:
print(huecos.groupby("empieza_tras_ps")["largo"].sum())

grandes = huecos[huecos["largo"] >= 50].copy()
n_pozo = {w: len(load_well(w)[0]) for w in grandes["well"].unique()}
grandes["n"] = grandes["well"].map(n_pozo)
grandes["en_borde"] = (grandes["inicio"] == 0) | (grandes["inicio"] + grandes["largo"] == grandes["n"])
print(grandes["en_borde"].value_counts())
print("pozos sin nada de GR:", (grandes["largo"] == grandes["n"]).sum())
print(grandes.sort_values("largo", ascending=False).head(8)[["well", "inicio", "largo", "n"]])

In [ ]:
filas = []
for w in wells:
    hz, tw, ps = load_well(w)
    out = clean_gr(hz)
    g = hz["GR"]
    filas.append({
        "well": w, "n": len(hz), "ps": ps,
        "hz_media": g.mean(), "hz_std": g.std(), "hz_min": g.min(), "hz_max": g.max(),
        "tw_media": tw["GR"].mean(), "tw_std": tw["GR"].std(),
        "tw_min": tw["GR"].min(), "tw_max": tw["GR"].max(),
        "nan_antes_ps": out["GR_mask"].iloc[:ps].mean(),
        "nan_desde_ps": out["GR_mask"].iloc[ps:].mean(),
    })
gr_stats = pd.DataFrame(filas)

print(gr_stats.drop(columns=["well", "n", "ps"]).describe().T[["mean", "min", "50%", "max"]].round(2))
print("pozos con GR < 0 (horizontal):", (gr_stats["hz_min"] < 0).sum())
print("pozos con GR < 0 (typewell):", (gr_stats["tw_min"] < 0).sum())

In [ ]:
gr_stats["dif_media"] = gr_stats["hz_media"] - gr_stats["tw_media"]
print(gr_stats["dif_media"].describe().round(2))
print("correlación hz_media vs tw_media:", round(gr_stats["hz_media"].corr(gr_stats["tw_media"]), 3))
print("pozos con hz_max > 300:", (gr_stats["hz_max"] > 300).sum(),
      "| tw_max > 300:", (gr_stats["tw_max"] > 300).sum())
print(gr_stats.sort_values("hz_max", ascending=False).head(5)[["well", "hz_media", "hz_std", "hz_max"]])

In [ ]:
hz, tw, ps = load_well("000d7d20")
out = clean_gr(hz)
suave = out["GR_clean"].rolling(9, center=True, min_periods=1).median()
suave = suave.where(out["GR_clean"].notna())   # no rellenar los huecos largos

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(out["MD"], out["GR_clean"], lw=0.6, label="GR_clean")
ax.plot(out["MD"], suave, lw=1.2, label="mediana móvil (9 ft)")
ax.axvline(out["MD"].iloc[ps], color="k", ls="--", label="PS")
ax.set_xlabel("MD (ft)")
ax.set_ylabel("GR")
ax.legend()
plt.show()

print("ruido (std de saltos / std total):",
      round(out["GR_clean"].diff().std() / out["GR_clean"].std(), 2))

In [ ]:
for w in ["a4f989c2", "197f8a5a", "d0cb63ae"]:
    hz, tw, ps = load_well(w)
    g = hz["GR"]
    i = g.idxmax()
    vecinos = g.iloc[max(i - 5, 0): i + 6].drop(index=i)
    z = (g - g.mean()) / g.std()
    print(w, "| máx:", round(g[i], 1), "en fila", i, "(PS:", ps, ")",
          "| mediana de vecinos:", round(vecinos.median(), 1),
          "| filas con z > 5:", int((z > 5).sum()))

ruido = []
for w in wells:
    hz, _, _ = load_well(w)
    g = clean_gr(hz)["GR_clean"]
    ruido.append(g.diff().std() / g.std())
print(pd.Series(ruido).describe().round(2))

## 4. Remuestreo del typewell

**Decisiones de remuestreo del typewell**

- Los 773 typewells tienen el TVT estrictamente creciente y no traen NaN en GR. El paso mediano es 0.5 ft en 653 pozos, 0.25 ft en 91, 1.0 ft en 21 y 0.10 ft en 8 (este último no figuraba en el notebook de referencia).
- La mayor parte de las diferencias de paso dentro de un typewell es redondeo de los decimales. Solo 3 typewells tienen un salto real de TVT (hasta 9 ft).
- Grilla común de 0.5 ft (`step`, parámetro de `resample_typewell`) con origen en el primer TVT de cada typewell. El GR se interpola linealmente; los saltos de más de 9 ft quedarían en NaN (ninguno lo supera). `Geology` no se interpola: cada punto toma la etiqueta del punto original anterior. No se pierde ninguna de las 43 etiquetas, y en promedio el 30.6 % de las filas de un typewell no tiene etiqueta.
- Costo de remuestrear: el error medio de GR al volver a los TVT originales equivale al 0.4 % del cambio típico entre puntos vecinos en los de 0.5 ft, 8 % en los de 0.10 ft, 11 % en los de 0.25 ft y 19 % en los de 1 ft.
- Cobertura: 13 pozos tienen un horizontal cuyo TVT sale del rango de su typewell, pero ninguno lo hace desde el PS (el tramo evaluado siempre queda cubierto en train). 10 de ellos comparten el mismo typewell (rango 11782.6 a 12285.1) y su horizontal baja entre 343 y 645 ft por debajo de su inicio. No se extrapola ni se inventan datos; la tabla maestra llevará las banderas `tw_cubre_pozo` y `tw_cubre_ps`.
- Hay 752 typewells distintos para 773 pozos; la tabla maestra guarda el typewell asignado.
- Picos agudos del typewell (por ejemplo en `e730c32b`, GR de 110 a 224 en 3 ft) pierden hasta unas 20 unidades de amplitud cuando un punto de la grilla cae entre dos muestras originales; en ese pozo solo 25 de 2188 puntos tienen error mayor a 5. El desfase máximo respecto a la grilla es de 0.25 ft, despreciable frente a los 15.9 ft de RMSE de referencia, y la posición de los picos se conserva.

In [ ]:
def resample_typewell(tw, step=0.5, max_gap=9.0):
    """Typewell a grilla regular de TVT (paso `step`, origen en su primer punto).

    GR: interpolación lineal; los huecos de más de max_gap pies quedan en NaN.
    Geology: etiqueta del punto original anterior (no se interpola texto).
    """
    t = tw["TVT"].to_numpy()
    grid = np.arange(t[0], t[-1] + 1e-9, step)

    j = np.clip(np.searchsorted(t, grid + 1e-6, side="right") - 1, 0, len(t) - 1)
    k = np.clip(j, 0, len(t) - 2)
    gap = t[k + 1] - t[k]

    gr = np.interp(grid, t, tw["GR"].to_numpy())
    gr[gap > max_gap + 1e-6] = np.nan
    geo = tw["Geology"].to_numpy()[j]
    return pd.DataFrame({"TVT": grid, "GR": gr, "Geology": geo})

### Verificaciones (se pueden omitir)

In [ ]:
filas = []
for w in wells:
    hz, tw, ps = load_well(w)
    t = tw["TVT"].to_numpy()
    paso = np.diff(t)
    filas.append({
        "well": w, "tw_n": len(tw),
        "tw_min": t.min(), "tw_max": t.max(),
        "paso_med": np.median(paso), "paso_min": paso.min(), "paso_max": paso.max(),
        "creciente": bool((paso > 0).all()),
        "hz_min": hz["TVT"].min(), "hz_max": hz["TVT"].max(),
        "hz_min_ps": hz["TVT"].iloc[ps:].min(), "hz_max_ps": hz["TVT"].iloc[ps:].max(),
        "gr_nan": tw["GR"].isna().mean(), "geol_nan": tw["Geology"].isna().mean(),
    })
tw_stats = pd.DataFrame(filas)
tw_stats["fuera"] = (tw_stats["hz_min"] < tw_stats["tw_min"]) | (tw_stats["hz_max"] > tw_stats["tw_max"])
tw_stats["fuera_ps"] = (tw_stats["hz_min_ps"] < tw_stats["tw_min"]) | (tw_stats["hz_max_ps"] > tw_stats["tw_max"])

print(tw_stats["paso_med"].round(3).value_counts())
print("pasos no uniformes:", ((tw_stats["paso_max"] - tw_stats["paso_min"]) > 1e-3).sum())
print("typewells con TVT no creciente:", (~tw_stats["creciente"]).sum())
print("typewell con GR vacío (media):", round(tw_stats["gr_nan"].mean(), 4))
print("pozos que salen del typewell | todo el pozo:", tw_stats["fuera"].sum(),
      "| solo desde el PS:", tw_stats["fuera_ps"].sum())
print(tw_stats[tw_stats["fuera"]][["well", "tw_min", "tw_max", "hz_min", "hz_max"]].round(1).head(15))

In [ ]:
import hashlib

tw_stats["dif_paso"] = tw_stats["paso_max"] - tw_stats["paso_min"]
print(tw_stats["dif_paso"].describe().round(3))
print("dif_paso > 0.02:", (tw_stats["dif_paso"] > 0.02).sum(),
      "| > 0.5:", (tw_stats["dif_paso"] > 0.5).sum())

w_irreg = tw_stats.loc[tw_stats["dif_paso"].idxmax(), "well"]
tw_i = load_well(w_irreg)[1]
print(w_irreg, "- pasos distintos (redondeados a 0.01):")
print(pd.Series(np.diff(tw_i["TVT"])).round(2).value_counts().head(8))

huellas = {}
for w in wells:
    tw_w = load_well(w)[1]
    huellas[w] = hashlib.md5(pd.util.hash_pandas_object(tw_w, index=False).to_numpy().tobytes()).hexdigest()
huellas = pd.Series(huellas)
print("typewells distintos:", huellas.nunique(), "de", len(huellas), "pozos")
print("pozos por typewell (los 5 más repetidos):", huellas.value_counts().head(5).tolist())

fuera = tw_stats[tw_stats["fuera"]].copy()
fuera["bajo_tw"] = (fuera["tw_min"] - fuera["hz_min"]).clip(lower=0).round(1)
fuera["sobre_tw"] = (fuera["hz_max"] - fuera["tw_max"]).clip(lower=0).round(1)
print(fuera[["well", "bajo_tw", "sobre_tw"]].to_string(index=False))
print("typewells distintos entre los 13:", huellas[fuera["well"]].nunique())

In [ ]:
tres = tw_stats.loc[tw_stats["dif_paso"] > 0.02, "well"].tolist()
etiquetas, malos, n_in, n_out, nan_gr, identicos = set(), [], 0, 0, 0, 0
for w in wells:
    tw = load_well(w)[1]
    r = resample_typewell(tw)
    etiquetas |= set(tw["Geology"].dropna().unique())
    n_in += len(tw); n_out += len(r); nan_gr += int(r["GR"].isna().sum())
    if not np.allclose(np.diff(r["TVT"]), 0.5):
        malos.append(w)
    if len(r) == len(tw) and np.allclose(r["TVT"], tw["TVT"]) and np.allclose(r["GR"], tw["GR"]):
        identicos += 1

print("pozos con paso != 0.5 tras remuestrear:", len(malos))
print("filas antes:", n_in, "| después:", n_out, "| NaN en GR:", nan_gr)
print("typewells idénticos al original:", identicos, "de", len(wells))
for w in tres:
    tw = load_well(w)[1]; r = resample_typewell(tw)
    print(w, "| filas", len(tw), "->", len(r), "| NaN en GR:", int(r["GR"].isna().sum()))
print(len(etiquetas), "etiquetas de Geology:", sorted(etiquetas)[:10])
print("NaN de Geology por pozo (media):", round(tw_stats["geol_nan"].mean(), 3))

In [ ]:
filas = []
for w, paso in zip(tw_stats["well"], tw_stats["paso_med"].round(2)):
    tw = load_well(w)[1]
    r = resample_typewell(tw)
    t = tw["TVT"].to_numpy()
    k = np.round((t - t[0]) / paso)
    filas.append({
        "well": w, "paso": paso,
        "d_filas": len(r) - len(tw),
        "deriva": np.abs((t - t[0]) - k * paso).max(),
        "gr_perdida": np.abs(np.interp(t, r["TVT"], r["GR"]) - tw["GR"].to_numpy()).max(),
        "geo_igual": set(tw["Geology"].dropna()) == set(r["Geology"].dropna()),
    })
cmp_tw = pd.DataFrame(filas)

print(cmp_tw.groupby("paso")[["deriva", "gr_perdida"]].agg(["median", "max"]).round(3))
print("pozos con distinto largo:", (cmp_tw["d_filas"] != 0).sum(),
      "| de ellos a 0.5 ft:", ((cmp_tw["paso"] == 0.5) & (cmp_tw["d_filas"] != 0)).sum())
print("pozos que pierden alguna etiqueta de Geology:", (~cmp_tw["geo_igual"]).sum())

In [ ]:
filas = []
for w, paso in zip(tw_stats["well"], tw_stats["paso_med"].round(2)):
    tw = load_well(w)[1]
    r = resample_typewell(tw)
    t, gr = tw["TVT"].to_numpy(), tw["GR"].to_numpy()
    err = np.abs(np.interp(t, r["TVT"], r["GR"]) - gr)
    filas.append({"well": w, "paso": paso,
                  "err_media": err.mean(), "err_max": err.max(),
                  "salto_nuevo": np.abs(np.diff(r["GR"])).mean()})
err_tw = pd.DataFrame(filas)
err_tw["err_rel"] = err_tw["err_media"] / err_tw["salto_nuevo"]

print(err_tw.groupby("paso")[["err_media", "salto_nuevo", "err_rel"]].median().round(3))
print(err_tw[err_tw["paso"] == 0.5].sort_values("err_max", ascending=False).head(5).round(2))

In [ ]:
w = "e730c32b"
tw = load_well(w)[1]
r = resample_typewell(tw)
t, gr = tw["TVT"].to_numpy(), tw["GR"].to_numpy()
err = np.abs(np.interp(t, r["TVT"], r["GR"]) - gr)
i = int(err.argmax())
print("fila del typewell:", i, "de", len(tw), "| TVT:", t[i], "| error:", round(err[i], 1))
print(tw.iloc[max(i - 3, 0): i + 4][["TVT", "GR"]])
print("puntos con error > 5:", int((err > 5).sum()), "de", len(tw))

## 5. Tabla maestra de pozos

## 6. Esquema de validación (folds espaciales)

## 7. Función de evaluación y validador del formato de predicción

## 8. Exportar salidas a `data/processed/p1/`

## Checklist de entrega

- [ ] Las salidas existen para los 773 pozos y se cargan sin errores.
- [ ] `rmse_by_well` reproduce 15.9 ft para la predicción "TVT constante = TVT(PS)" (valor de referencia del notebook 00).
- [ ] Los folds no mezclan pozos vecinos (< 500 ft) entre train y validación.
- [ ] Documentadas las decisiones de limpieza de GR y remuestreo.